# Flipkart Product Links & Reviews Extraction

## Overview
This notebook extracts product links from Flipkart search results and prepares a structured product table with links for individual product review scraping.

### Workflow
1. **Extract Product Links**: Scrape search result pages to get links to all product pages
2. **Build Product Table**: Create a DataFrame with product URLs and basic info
3. **Prepare for Review Scraping**: Structure data for scraping individual product pages

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 2. Extract Product Links from Search Results

The `extract_flipkart_product_links.py` script has already extracted 300+ product links from 20 pages of Flipkart mobile search results.

In [2]:
# Read the extracted product links
links_file = "product_links.csv"

if Path(links_file).exists():
    df_links = pd.read_csv(links_file)
    print(f"✅ Loaded {len(df_links)} product links from {links_file}")
    print(f"\nColumns: {df_links.columns.tolist()}")
    print(f"\nFirst 5 links:")
    print(df_links.head())
else:
    print(f"❌ File not found: {links_file}")
    print(f"   Run: python extract_flipkart_product_links.py --pages 20")

✅ Loaded 304 product links from product_links.csv

Columns: ['product_url']

First 5 links:
                                         product_url
0  https://www.flipkart.com/ai-nova-2-5g-blue-64-...
1  https://www.flipkart.com/ai-nova-2-5g-green-64...
2  https://www.flipkart.com/ai-nova-2-pro-5g-blue...
3  https://www.flipkart.com/ai-nova-2-ultra-5g-bl...
4  https://www.flipkart.com/ai-nova-2-ultra-5g-bl...


## 3. Build Product Table Structure

Create a structured DataFrame that will hold:
- Product URL (for scraping reviews)
- Product ID (extracted from URL)
- Placeholder columns for review data and sentiment analysis

In [3]:
# Create product table with structure for reviews
df_products = df_links.copy()

# Extract product ID from URL (the part after /p/)
df_products['product_id'] = df_products['product_url'].str.extract(r'/p/([^/?]+)')

# Add placeholder columns for review data
df_products['reviews_count'] = 0
df_products['average_rating'] = np.nan
df_products['reviews_scraped'] = False

# Reorder columns
df_products = df_products[['product_id', 'product_url', 'reviews_count', 'average_rating', 'reviews_scraped']]

print(f"✅ Created product table with {len(df_products)} rows")
print(f"\nTable structure:")
print(df_products.info())
print(f"\nFirst 10 rows:")
print(df_products.head(10))

✅ Created product table with 304 rows

Table structure:
<class 'pandas.DataFrame'>
RangeIndex: 304 entries, 0 to 303
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   product_id       304 non-null    str    
 1   product_url      304 non-null    str    
 2   reviews_count    304 non-null    int64  
 3   average_rating   0 non-null      float64
 4   reviews_scraped  304 non-null    bool   
dtypes: bool(1), float64(1), int64(1), str(2)
memory usage: 9.9 KB
None

First 10 rows:
         product_id                                        product_url  \
0  itm20a7387042b2c  https://www.flipkart.com/ai-nova-2-5g-blue-64-...   
1  itm93c2428cc4241  https://www.flipkart.com/ai-nova-2-5g-green-64...   
2  itmfcb5acce99b5c  https://www.flipkart.com/ai-nova-2-pro-5g-blue...   
3  itm41a37f63787e9  https://www.flipkart.com/ai-nova-2-ultra-5g-bl...   
4  itm2d6be0d65d846  https://www.flipkart.com/ai-nova-2-ultra-5g-bl... 

## 4. Save Product Table for Review Scraping

In [4]:
# Save the product table
output_file = "flipkart_products_for_reviews.csv"
df_products.to_csv(output_file, index=False, encoding='utf-8')

print(f"✅ Saved product table to {output_file}")
print(f"   Rows: {len(df_products)}")
print(f"   Size: {Path(output_file).stat().st_size / 1024:.2f} KB")

# Show summary
print(f"\n📊 Product Table Summary:")
print(f"   - Total products: {len(df_products)}")
print(f"   - Unique product IDs: {df_products['product_id'].nunique()}")
print(f"   - Ready for review scraping: ✅")

✅ Saved product table to flipkart_products_for_reviews.csv
   Rows: 304
   Size: 31.36 KB

📊 Product Table Summary:
   - Total products: 304
   - Unique product IDs: 220
   - Ready for review scraping: ✅


## 5. Next Steps: Individual Product Review Scraping

### Option A: Use the Review Scraper Script
```bash
python scrape_product_reviews.py --input flipkart_products_for_reviews.csv --max 50 --delay 2.0
```

### Option B: Manual Review Scraping with Selenium
For handling JavaScript-rendered content and avoiding reCAPTCHA:

```python
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
for idx, row in df_products.iterrows():
    url = row['product_url']
    driver.get(url)
    # Extract reviews, ratings, etc.
```

### Option C: Batch Process with Delays
Extract reviews in batches with longer delays to avoid blocking:

```python
batch_size = 10
delay_between_batches = 300  # 5 minutes
```

## 6. Final Combined Product Table Structure

This will be the structure of the final table after review scraping:

In [5]:
# Show the final table structure that will be created after review scraping
final_columns = [
    'product_id',              # Unique product identifier
    'product_url',             # URL for scraping
    'product_name',            # Product title
    'price',                   # Price
    'rating',                  # Overall rating
    'reviews_count',           # Total reviews
    'reviewer_name',           # Individual reviewer
    'review_rating',           # Individual review rating
    'review_title',            # Review title
    'review_text',             # Full review text
    'sentiment_label',         # Vader/Roberta sentiment (positive/negative/neutral)
    'sentiment_score',         # Sentiment confidence score
    'helpful_count',           # People found helpful
]

print("Final Combined Product Table Structure:")
print("="*50)
for i, col in enumerate(final_columns, 1):
    print(f"{i:2d}. {col}")

print(f"\n📝 Notes:")
print(f"   - Each row = 1 review for 1 product")
print(f"   - Multiple reviews per product = multiple rows")
print(f"   - Total rows = product_count × average_reviews_per_product")
print(f"   - Current products ready: {len(df_products)}")

Final Combined Product Table Structure:
 1. product_id
 2. product_url
 3. product_name
 4. price
 5. rating
 6. reviews_count
 7. reviewer_name
 8. review_rating
 9. review_title
10. review_text
11. sentiment_label
12. sentiment_score
13. helpful_count

📝 Notes:
   - Each row = 1 review for 1 product
   - Multiple reviews per product = multiple rows
   - Total rows = product_count × average_reviews_per_product
   - Current products ready: 304


## 7. Quick Statistics

In [6]:
print("📊 Product Links Extraction Summary")
print("="*50)
print(f"Total unique products extracted: {len(df_products)}")
print(f"From search results: {len(df_products) // 30} pages (approx. 30 products/page)")
print(f"\n💾 Files generated:")
print(f"   1. product_links.csv - All extracted links")
print(f"   2. flipkart_products_for_reviews.csv - Structured product table")
print(f"\n🔄 Next action:")
print(f"   Run: python scrape_product_reviews.py --input flipkart_products_for_reviews.csv")
print(f"   This will scrape reviews from all {len(df_products)} products and add sentiment analysis")

📊 Product Links Extraction Summary
Total unique products extracted: 304
From search results: 10 pages (approx. 30 products/page)

💾 Files generated:
   1. product_links.csv - All extracted links
   2. flipkart_products_for_reviews.csv - Structured product table

🔄 Next action:
   Run: python scrape_product_reviews.py --input flipkart_products_for_reviews.csv
   This will scrape reviews from all 304 products and add sentiment analysis
